# Cross-Industry Accelerator — Create Semantic Model
### Auto-Generate Power BI Semantic Model from Lakehouse or Warehouse

This notebook creates a **Power BI Semantic Model** (dataset) by:

1. Reading the dynamic table/schema discovery from `00_Industry_Config`
2. Letting you choose the **data source**: Lakehouse (Delta tables) or Warehouse (SQL tables)
3. Building a TMSL (Tabular Model Scripting Language) definition with:
   - All dimension and fact tables as model tables
   - Auto-detected relationships (FK columns matching dim PK patterns)
   - Auto-generated measures for numeric fact columns (SUM, AVG, COUNT)
   - Date hierarchies on detected date columns
4. Creating the semantic model in Fabric via the REST API

> **Prerequisites:**
> 1. Run `02_Load_Lakehouse.ipynb` and/or `03_Load_Warehouse.ipynb` so tables exist
> 2. The Fabric workspace must have Power BI capacity assigned

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# READ TABLE SCHEMAS FROM CSVs
# ════════════════════════════════════════════════════════════════════════

import pandas as pd
from pyspark.sql.types import (
    StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, DecimalType
)

# Spark type → TMSL data type mapping
SPARK_TO_TMSL = {
    StringType:    "string",
    IntegerType:   "int64",
    LongType:      "int64",
    FloatType:     "double",
    DoubleType:    "double",
    BooleanType:   "boolean",
    DateType:      "dateTime",
    TimestampType: "dateTime",
    DecimalType:   "decimal",
}

# Read schemas for all tables (dim + fact + event + stream)
# ── Choose data source: "lakehouse" or "warehouse" ──────────────────
SEMANTIC_MODEL_SOURCE = "warehouse"   # "lakehouse" → Delta tables  |  "warehouse" → SQL tables

print(f"Semantic Model Source: {SEMANTIC_MODEL_SOURCE.upper()}")
if SEMANTIC_MODEL_SOURCE == "lakehouse":
    print(f"  → Tables will be bound to Lakehouse: {LAKEHOUSE_NAME}")
else:
    print(f"  → Tables will be bound to Warehouse: {WAREHOUSE_NAME}")

ALL_TABLES = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT + STREAM_TABLES
table_schemas = {}

print(f"Reading schemas for {len(ALL_TABLES)} tables...\n")
for table_name in ALL_TABLES:
    csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
    try:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))
        table_schemas[table_name] = df.schema
        print(f"  ✓ {table_name:<45} {len(df.schema.fields)} columns")
    except Exception as e:
        print(f"  ✗ {table_name:<45} ERROR: {e}")

print(f"\n✅ Schemas loaded for {len(table_schemas)} tables.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GENERATE BIM MODEL DEFINITION (proven Fabric format)
# ════════════════════════════════════════════════════════════════════════

import json
import sempy.fabric as fabric
import requests as req_lib

# Get the Warehouse SQL endpoint
workspace_id = fabric.get_workspace_id()
pbi_token = notebookutils.credentials.getToken('pbi')
api_headers = {"Authorization": f"Bearer {pbi_token}", "Content-Type": "application/json"}
BASE_URL = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

items_df = fabric.list_items()
wh_match = items_df[(items_df["Type"].isin(["DataWarehouse", "Warehouse"])) &
                    (items_df["Display Name"] == WAREHOUSE_NAME)]
wh_id = wh_match.iloc[0].Id
wh_resp = req_lib.get(f"{BASE_URL}/warehouses/{wh_id}", headers=api_headers)
SQL_ENDPOINT = wh_resp.json().get("properties", {}).get("connectionString", "")
print(f"SQL Endpoint: {SQL_ENDPOINT}")

# Spark type -> BIM data type
SPARK_TO_BIM = {
    StringType: "string", IntegerType: "int64", LongType: "int64",
    FloatType: "double", DoubleType: "double", BooleanType: "boolean",
    DateType: "string", TimestampType: "string", DecimalType: "double",
}

# Shared M expression for warehouse connection
WAREHOUSE_EXPR = f'let\n    database = Sql.Database("{SQL_ENDPOINT}", "{WAREHOUSE_NAME}")\nin\n    database'

# Build tables
bim_tables = []
all_measures_count = 0

for table_name, schema in table_schemas.items():
    columns = []
    for f in schema.fields:
        columns.append({
            "name": f.name,
            "dataType": SPARK_TO_BIM.get(type(f.dataType), "string"),
            "sourceColumn": f.name,
            "summarizeBy": "none",
        })

    partition_expr = (
        f'let\n'
        f'    Source = DatabaseQuery,\n'
        f'    Data = Source{{[Schema="dbo",Item="{table_name}"]}}[Data]\n'
        f'in\n'
        f'    Data'
    )

    table_def = {
        "name": table_name,
        "columns": columns,
        "partitions": [{
            "name": table_name,
            "mode": "directQuery",
            "source": {"type": "m", "expression": partition_expr},
        }],
    }

    # Add measures to fact tables (prefixed with table name to avoid duplicates)
    if not table_name.startswith("dim_"):
        measures = []
        short_name = table_name.replace("fact_", "").replace("stream_", "").replace("_", " ").title()
        for f in schema.fields:
            if isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType)) and not f.name.lower().endswith("_id"):
                label = f.name.replace("_", " ").title()
                measures.append({"name": f"{short_name} Total {label}", "expression": f"SUM('{table_name}'[{f.name}])", "formatString": "#,0"})
                measures.append({"name": f"{short_name} Avg {label}", "expression": f"AVERAGE('{table_name}'[{f.name}])", "formatString": "0.0"})
                all_measures_count += 2
        if measures:
            table_def["measures"] = measures

    bim_tables.append(table_def)

# Auto-detect relationships
def detect_relationships(tbl_schemas, dim_tables):
    rels = []
    dim_pks = {}
    for dim in dim_tables:
        if dim in tbl_schemas:
            id_cols = [f.name for f in tbl_schemas[dim].fields if f.name.lower().endswith("_id")]
            if id_cols:
                dim_pks[dim] = id_cols[0]
    fact_tables = [t for t in tbl_schemas if t not in dim_tables]
    for i, fact in enumerate(fact_tables):
        fact_cols = [f.name for f in tbl_schemas[fact].fields]
        for dim, pk in dim_pks.items():
            if pk in fact_cols:
                rels.append({
                    "name": f"rel_{len(rels):03d}_{fact}_{dim}",
                    "fromTable": fact,
                    "fromColumn": pk,
                    "toTable": dim,
                    "toColumn": pk,
                })
    return rels

relationships = detect_relationships(table_schemas, DIM_TABLES)

# Assemble BIM
bim = {
    "compatibilityLevel": 1604,
    "model": {
        "culture": "en-US",
        "defaultPowerBIDataSourceVersion": "powerBI_V3",
        "sourceQueryCulture": "en-US",
        "tables": bim_tables,
        "expressions": [{"name": "DatabaseQuery", "kind": "m", "expression": WAREHOUSE_EXPR}],
        "relationships": relationships,
        "annotations": [{"name": "__PBI_TimeIntelligenceEnabled", "value": "0"}],
    },
}

print(f"\nBIM model definition generated:")
print(f"   Tables:         {len(bim_tables)}")
print(f"   Relationships:  {len(relationships)}")
print(f"   Measures:       {all_measures_count}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PREVIEW: AUTO-DETECTED RELATIONSHIPS
# ════════════════════════════════════════════════════════════════════════

if relationships:
    print(f"Auto-detected {len(relationships)} relationships:\n")
    rel_df = pd.DataFrame(relationships)
    print(rel_df[["fromTable", "fromColumn", "toTable", "toColumn"]].to_string(index=False))
else:
    print("No auto-detected relationships (check FK naming conventions: columns ending with _id).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE SEMANTIC MODEL IN FABRIC (proven format from healthcare use case)
# ════════════════════════════════════════════════════════════════════════

import base64
import time

def to_base64(obj):
    return base64.b64encode(json.dumps(obj, indent=2).encode("utf-8")).decode("ascii")

pbism = {"version": "1.0"}
definition = {
    "parts": [
        {"path": "definition.pbism", "payload": to_base64(pbism), "payloadType": "InlineBase64"},
        {"path": "model.bim", "payload": to_base64(bim), "payloadType": "InlineBase64"},
    ]
}

# Check if model already exists
existing_id = None
try:
    sm_resp = req_lib.get(f"{BASE_URL}/semanticModels", headers=api_headers)
    if sm_resp.status_code == 200:
        for item in sm_resp.json().get("value", []):
            if item["displayName"] == SEMANTIC_MODEL_NAME:
                existing_id = item["id"]
                break
except Exception:
    pass

if existing_id:
    print(f"Model '{SEMANTIC_MODEL_NAME}' exists (ID: {existing_id}). Updating definition...")
    resp = req_lib.post(
        f"{BASE_URL}/semanticModels/{existing_id}/updateDefinition",
        headers=api_headers,
        json={"definition": definition},
    )
else:
    print(f"Creating semantic model: {SEMANTIC_MODEL_NAME}...")
    resp = req_lib.post(
        f"{BASE_URL}/items",
        headers=api_headers,
        json={
            "displayName": SEMANTIC_MODEL_NAME,
            "type": "SemanticModel",
            "definition": definition,
        },
    )

print(f"  HTTP {resp.status_code}")

if resp.status_code == 200:
    print(f"  Definition updated in-place.")
elif resp.status_code == 201:
    result = resp.json()
    print(f"  Created: {result.get('displayName')} (ID: {result.get('id')})")
elif resp.status_code == 202:
    op_url = resp.headers.get("Location") or resp.headers.get("Operation-Location", "")
    if op_url:
        print(f"  Provisioning...")
        for _ in range(30):
            time.sleep(4)
            poll = req_lib.get(op_url, headers=api_headers)
            if poll.status_code == 200:
                status = poll.json().get("status", "")
                print(f"    Status: {status}")
                if status in ("Succeeded", "succeeded"):
                    print(f"  Model created successfully.")
                    break
                elif status in ("Failed", "failed"):
                    print(f"  FAILED: {json.dumps(poll.json(), indent=2)}")
                    break
        else:
            print("  Timed out.")
    else:
        print("  Accepted (202). Waiting...")
        time.sleep(10)
else:
    print(f"  Error: {resp.text[:500]}")

# Verify
time.sleep(3)
verify_id = None
try:
    sm_resp = req_lib.get(f"{BASE_URL}/semanticModels", headers=api_headers)
    for item in sm_resp.json().get("value", []):
        if item["displayName"] == SEMANTIC_MODEL_NAME:
            verify_id = item["id"]
            break
except Exception:
    pass

if verify_id:
    print(f"\n  Semantic model in workspace: {verify_id}")
    print(f"  Open in Power BI to build reports.")
else:
    print(f"\n  Model not found. Check Fabric portal.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SEMANTIC MODEL SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*65}")
print(f"SEMANTIC MODEL SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*65}")
print(f"\n  Model Name:      {SEMANTIC_MODEL_NAME}")

# Determine data source (warehouse is default)
source_type = SEMANTIC_MODEL_SOURCE if 'SEMANTIC_MODEL_SOURCE' in dir() else 'warehouse'
source_name = LAKEHOUSE_NAME if source_type == 'lakehouse' else WAREHOUSE_NAME
print(f"  Data Source:     {source_name} ({source_type})")
print(f"")
print(f"  Star Schema:")
print(f"    Dimensions:    {len([t for t in table_schemas if t.startswith('dim_')])}")
print(f"    Facts:         {len([t for t in table_schemas if not t.startswith('dim_')])}")
print(f"    Relationships: {len(relationships)}")
print(f"    Measures:      {all_measures_count}")
print(f"")
print(f"  Tables in model:")
for t in bim_tables:
    cat = "DIM" if t['name'].startswith('dim_') else "FACT"
    m_count = len(t.get('measures', []))
    m_text = f"  ({m_count} measures)" if m_count > 0 else ""
    print(f"    [{cat}] {t['name']:<40} {len(t['columns'])} cols{m_text}")

print(f"\n✅ Semantic model creation complete.")
print(f"   Next: Run 05_Create_Ontology.ipynb to create the industry ontology.")